# Notebook 07 — Training data inspection

Inspect and understand the data your model will be (or was) trained on.
**No model or GPU needed — just Python and your data files.**

Good practice before training: always look at your data. Common issues:
- Very short answers (model learns to give 2-word responses)
- Very long sequences (will be truncated and lose information)
- Duplicates (inflates accuracy metrics)
- Format errors (missing `<|endoftext|>`, wrong field names)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════

# Path to your generated training data
SYNTHETIC_JSONL = "../teaching_assistant/data/synthetic_qa.jsonl"

# Whether to also load from HuggingFace datasets
# (requires: pip install datasets, and internet connection)
LOAD_HF_DATASETS = False
MAX_HF_EXAMPLES  = 100   # only load this many to keep things fast

# Token budget (should match your model's block_size)
MAX_LENGTH = 512

In [ ]:
# ═══════════════════════════════════════════════════════════
# LOAD DATA
# ═══════════════════════════════════════════════════════════
import json
from pathlib import Path

all_examples = []

# Load synthetic QA (from your course material + GPT generation)
if Path(SYNTHETIC_JSONL).exists():
    with open(SYNTHETIC_JSONL) as f:
        for line in f:
            line = line.strip()
            if line:
                all_examples.append(json.loads(line))
    print(f"Loaded {len(all_examples)} synthetic examples from {SYNTHETIC_JSONL}")
else:
    print(f"No synthetic data found at {SYNTHETIC_JSONL}")
    print("Run data_generation.generate_all_modes() first, or create dummy data below.")

# Load from HuggingFace (optional)
if LOAD_HF_DATASETS:
    from datasets import load_dataset
    ds = load_dataset("prsdm/Machine-Learning-QA-dataset", split="train")
    for ex in list(ds)[:MAX_HF_EXAMPLES]:
        all_examples.append({
            "type":     "qa",
            "question": ex.get("question", ""),
            "answer":   ex.get("answer", ""),
            "source":   "prsdm_hf",
        })
    print(f"Added {MAX_HF_EXAMPLES} HuggingFace examples")

print(f"\nTotal examples: {len(all_examples)}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# TYPE BREAKDOWN
# How many examples of each type (qa / explanation / review)?
# ═══════════════════════════════════════════════════════════
from collections import Counter

type_counts  = Counter(ex.get("type", "qa") for ex in all_examples)
source_counts = Counter(ex.get("source", "unknown") for ex in all_examples)

print("By type:")
for t, n in type_counts.most_common():
    pct = 100 * n / len(all_examples)
    bar = "█" * (n * 40 // len(all_examples))
    print(f"  {t:<15} {bar:<40} {n:>5} ({pct:.0f}%)")

print("\nBy source:")
for s, n in source_counts.most_common():
    print(f"  {s:<30} {n:>5}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# TOKEN LENGTH DISTRIBUTION
# See how many examples will be truncated at MAX_LENGTH.
# Truncated examples lose their ending — if MAX_LENGTH is too
# small, your answers get cut off during training.
# ═══════════════════════════════════════════════════════════
import tiktoken
enc = tiktoken.get_encoding("gpt2")

lengths = []
for ex in all_examples:
    # Build the training string (same format as dataset.py)
    text = ex.get("text", "")
    if not text:
        q = ex.get("question", ex.get("request", ""))
        a = ex.get("answer", ex.get("explanation", ex.get("review", "")))
        text = f"Question: {q}\nAnswer: {a}<|endoftext|>"
    lengths.append(len(enc.encode(text, allowed_special={"<|endoftext|>"})))

import statistics
print(f"Token length statistics:")
print(f"  Min:    {min(lengths):>6}")
print(f"  Median: {statistics.median(lengths):>6.0f}")
print(f"  Mean:   {statistics.mean(lengths):>6.0f}")
print(f"  Max:    {max(lengths):>6}")
print(f"  Stdev:  {statistics.stdev(lengths):>6.0f}")
n_over = sum(1 for l in lengths if l > MAX_LENGTH)
print(f"\n  Over MAX_LENGTH ({MAX_LENGTH}): {n_over} examples ({100*n_over//len(lengths)}%) — will be dropped")

# Histogram (ASCII)
print("\nLength histogram:")
bins = [0, 50, 100, 150, 200, 300, 400, 512, 700, 9999]
bin_labels = [f"{bins[i]}-{bins[i+1]-1}" for i in range(len(bins)-1)]
bin_counts = [0] * (len(bins)-1)
for l in lengths:
    for i in range(len(bins)-1):
        if bins[i] <= l < bins[i+1]:
            bin_counts[i] += 1
            break
max_count = max(bin_counts) or 1
for label, count in zip(bin_labels, bin_counts):
    bar = "█" * (count * 40 // max_count)
    print(f"  {label:>10}  {bar:<40} {count}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# SAMPLE: look at 5 random examples
# Always read your data — catch formatting issues early.
# ═══════════════════════════════════════════════════════════
import random
samples = random.sample(all_examples, min(5, len(all_examples)))

for i, ex in enumerate(samples):
    print(f"\n{'═'*60}")
    print(f"Example {i+1}  |  type={ex.get('type','qa')}  |  source={ex.get('source','?')}")
    print('─'*60)
    if "text" in ex:
        # Pre-formatted training string
        print(ex["text"][:400] + ("..." if len(ex["text"]) > 400 else ""))
    else:
        q_key = "question" if "question" in ex else "request"
        a_key = "answer" if "answer" in ex else "explanation"
        print(f"Q: {ex.get(q_key, '?')[:200]}")
        print(f"A: {ex.get(a_key, '?')[:200]}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# QUALITY CHECKS
# Flag potential issues before training.
# ═══════════════════════════════════════════════════════════

issues = []

for i, ex in enumerate(all_examples):
    a_key = "answer" if "answer" in ex else "explanation"
    q_key = "question" if "question" in ex else "request"
    q = ex.get(q_key, "")
    a = ex.get(a_key, "")

    if not q or len(q.strip()) < 10:
        issues.append((i, "question too short", q[:50]))
    if not a or len(a.strip()) < 10:
        issues.append((i, "answer too short",   a[:50]))
    if len(a.split()) < 5:
        issues.append((i, "answer < 5 words",   a[:80]))

if issues:
    print(f"Found {len(issues)} potential issues:")
    for idx, reason, preview in issues[:20]:   # show first 20
        print(f"  Example {idx:4d}: {reason:<25} | '{preview}'")
    if len(issues) > 20:
        print(f"  ... and {len(issues)-20} more")
else:
    print("No quality issues found — data looks clean!")

# Check for duplicates (exact question matches)
questions = [ex.get("question", ex.get("request", "")) for ex in all_examples]
n_unique = len(set(questions))
n_dupes  = len(questions) - n_unique
print(f"\nDuplicate questions: {n_dupes} ({100*n_dupes//len(questions)}%)")
if n_dupes > 0:
    print("Consider deduplicating before training.")